# Raster2Seq on our floor plans

**What this answers.** Our stage 5 reads a plan by finding walls and flooding the space
between them. Raster2Seq skips that: it predicts **labelled room polygons directly** from
the raster image — kitchen, living room, bedroom, bath, entry, storage — plus every door
and window as its own instance. If it works on estate-agent plans it replaces the
watershed, the caption seeding *and* the open-plan splitting problem in one move.

It reports 88.7 room F1 on CubiCasa5K, MIT licence, published checkpoint. The only reason
we have not already tried it is that it needs a GPU and two CUDA extensions compiled from
source. That is what this notebook is for.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Free tier is fine.

**What you get back:** a zip containing, for every one of our 25 plans, the predicted room
polygons in our own pixel coordinates, a side-by-side picture, and the same
outline-on-wall score the current engine is measured with — so the comparison is
like-for-like rather than vibes.

Roughly 25 minutes, most of it compiling.

## 1. Check the GPU

If this says "no GPU", stop and change the runtime type — everything below needs one.

In [ ]:
import subprocess
import torch

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no nvidia-smi on this runtime")
print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun"

## 2. Get the code

Raster2Seq vendors its own copy of detectron2, so there is no separate detectron2 install
to fight with.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Cornell-VAILab/Raster2Seq.git 2>/dev/null || echo "already cloned"
%cd /content/Raster2Seq
!ls

## 3. Dependencies — without breaking the runtime

**Do not install `requirements.txt`.** It pins `numpy==1.26.4` from the authors' 2024
environment, and Colab ships a *matched set* of binary wheels — NumPy, OpenCV, SciPy,
Matplotlib, pandas — all compiled against each other. Move NumPy and every one of them
starts raising

```
ValueError: numpy.dtype size changed, may indicate binary incompatibility.
            Expected 96 from C header, got 88 from PyObject
```

which is a broken runtime, not a warning.

So: pin NumPy to whatever is already installed. pip then has to solve around it, and if
some package genuinely cannot live with it, pip says so instead of quietly wrecking the
session. Only the handful of packages Colab does not already have get installed.

> **If you already ran an earlier version of this cell and hit that error**, the downgrade
> cannot be reliably undone in place. **Runtime → Disconnect and delete runtime**, then run
> this notebook from the top.

In [ ]:
import subprocess
import sys

import numpy

# Hold NumPy exactly where Colab put it.
PIN = f"numpy=={numpy.__version__}"

# Everything on the inference path that Colab does not ship. cv2, matplotlib,
# imageio, PIL, plotly, shapely, tqdm and huggingface_hub are already there, and
# asking pip for them again risks a rebuild we do not want.
NEEDED = ["descartes", "omegaconf", "fvcore", "pycocotools"]


def pip_install(*packages):
    """Install, with NumPy held where it is. Use this everywhere in this notebook:
    one unpinned install anywhere is enough to break every binary wheel."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIN, *packages],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
        raise SystemExit(f"pip could not install {packages} around {PIN} — see above")


print(f"holding {PIN}")
pip_install(*NEEDED)

# Check in a *fresh* interpreter: this process still holds the modules it imported
# before the install, so it cannot see a break.
check = subprocess.run(
    [sys.executable, "-c",
     "import numpy, cv2, matplotlib, shapely, plotly, imageio, torch;"
     "print(numpy.__version__, cv2.__version__, matplotlib.__version__)"],
    capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1200:])
    raise SystemExit(
        "a binary module no longer imports. Runtime > Disconnect and delete runtime, "
        "then run from the top.")
print("numpy / cv2 / matplotlib:", check.stdout.strip(), "— all still import")

## 4. Make the repository run on current libraries

**This is the cell that fails on a stock Colab.** Raster2Seq targets a 2024 environment and
four things have moved since. Each has an exact modern spelling, so all four are renames
rather than reimplementations:

| what the code says | what it needs | why |
|---|---|---|
| `AT_DISPATCH_FLOATING_TYPES(value.type(), …)` | `value.scalar_type()` | `Tensor::type()` was removed; the macro wants a `ScalarType` |
| `AT_ASSERTM(value.type().is_cuda(), …)` | `value.is_cuda()` | same removal; it only ever wanted a bool |
| `from matplotlib.cm import get_cmap` | `matplotlib.colormaps[name]` | gone in Matplotlib 3.9 |
| `torch.load(ckpt, map_location=…)` | `…, weights_only=False` | PyTorch 2.6 flipped the default to `True`, which refuses a training checkpoint |

The first two are why `nvcc` dies in a wall of template noise ending in
`failed with exit code 2`. The other two would have surfaced later, one at import and one
at checkpoint load.

Applied below, then built. Every replacement is guarded against matching its own output,
so re-running after a failed build is safe.

**If the build still fails, that is survivable** — §4c falls back to Deformable-DETR's own
pure-PyTorch implementation of the same operator. Slower and inference-only, which is all
this notebook does. Checked against an independent implementation of the operator's
definition, it agrees to 3e-16.

In [ ]:
import re
from pathlib import Path

REPO = Path("/content/Raster2Seq")

CMAP_OLD = "from matplotlib.cm import get_cmap"
CMAP_MARK = "# patched: matplotlib >= 3.9"
CMAP_NEW = f"""try:
    {CMAP_OLD}
except ImportError:                      {CMAP_MARK} removed it
    from matplotlib import colormaps

    def get_cmap(name=None, lut=None):
        return colormaps[name]"""


def modernise(root: Path) -> dict:
    """Rename what current PyTorch and Matplotlib no longer provide.

    Every replacement is guarded against matching its own output, so re-running
    changes nothing further -- which matters, because a failed build is exactly
    when someone re-runs the cell.
    """
    hits = {}
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix in (".cu", ".cuh", ".cpp", ".h", ".hpp"):
            src = original = path.read_text()
            src, a = re.subn(r"(AT_DISPATCH_\w+\(\s*)(\w+)\.type\(\)",
                             r"\1\2.scalar_type()", src)
            src, b = re.subn(r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()", src)
            n = a + b
        elif path.suffix == ".py":
            src = original = path.read_text()
            n = 0
            if CMAP_MARK not in src and CMAP_OLD in src:
                src = src.replace(CMAP_OLD, CMAP_NEW, 1)
                n += 1
            src, c = re.subn(
                r"torch\.load\(([^)]*?map_location=[^)]*?)\)",
                lambda m: (m.group(0) if "weights_only" in m.group(1)
                           else f"torch.load({m.group(1)}, weights_only=False)"),
                src)
            n += c
        else:
            continue
        if src != original:
            path.write_text(src)
            hits[str(path.relative_to(root))] = n
    return hits


edits = modernise(REPO)
for name, n in sorted(edits.items()):
    print(f"  {name}: {n}")
print(f"patched {len(edits)} files" if edits else "nothing left to patch")

stale = [p.name for p in REPO.rglob("*.cu") if ".type()" in p.read_text()]
print("remaining .type() call sites:", stale or "none")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

REPO = Path("/content/Raster2Seq")
major, minor = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"building for compute capability {major}.{minor} — {torch.cuda.get_device_name(0)}\n")


def build(where: Path, cmd: list) -> bool:
    r = subprocess.run(cmd, cwd=where, capture_output=True, text=True, env=os.environ)
    ok = r.returncode == 0
    print(f"{where.name}: {'built' if ok else 'FAILED'}")
    if not ok:
        for line in (r.stdout + r.stderr).splitlines()[-12:]:
            print("   ", line)
    return ok


# `pip install .` rather than the repo's `setup.py build install`: that installs an
# egg and registers it through easy-install.pth, which a kernel started before the
# build has never read -- so the extension builds and the very next cell still
# cannot import it. pip puts a plain module in site-packages instead.
# --no-build-isolation so it builds against the torch already installed here.
attn_ok = build(REPO / "models" / "ops",
                [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "."])
ras_ok = build(REPO / "diff_ras",
               [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "."])
if not ras_ok:
    print("\n  the rasteriser is only used by polygon refinement, which we disable — fine")

### 4c. Confirm the operator works, one way or the other

If the extension built, this imports it and moves on. If it did not, this rewrites
`ms_deform_attn_func.py` so the same operator runs in pure PyTorch, and says so plainly —
so you always know which of the two you ran, rather than discovering it from a
mysteriously slow or subtly wrong result later.

In [ ]:
import importlib
import importlib.util
import site
import sys
from pathlib import Path

import torch

REPO = Path("/content/Raster2Seq")
sys.path.insert(0, str(REPO))
FUNC = REPO / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
UPSTREAM_IMPORT = "import MultiScaleDeformableAttention as MSDA"

SHIM = """
try:
    import MultiScaleDeformableAttention as MSDA
except ImportError:
    # The CUDA extension did not build. Deformable-DETR ships a pure-PyTorch
    # implementation of this exact operator -- ms_deform_attn_core_pytorch, further
    # down this file -- so route the forward pass through it. Inference only:
    # backward needs the extension, and this notebook never trains.
    class _PurePythonMSDA:
        @staticmethod
        def ms_deform_attn_forward(value, value_spatial_shapes, value_level_start_index,
                                   sampling_locations, attention_weights, im2col_step):
            return ms_deform_attn_core_pytorch(
                value, value_spatial_shapes, sampling_locations, attention_weights)

        @staticmethod
        def ms_deform_attn_backward(*_args, **_kwargs):
            raise RuntimeError(
                "MultiScaleDeformableAttention did not build, so only inference is "
                "available here. Training needs the compiled extension.")

    MSDA = _PurePythonMSDA()
"""


def find_extension():
    """Import the compiled op, refreshing the path before giving up on it.

    A package installed while this kernel was already running is invisible to it
    until the import caches are dropped -- and an egg-style install is invisible
    until its .pth file is read too. Both are worth trying before concluding the
    build failed, because concluding wrongly silently costs a 10x slowdown.
    """
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        pass
    site.main()
    roots = list(site.getsitepackages())
    try:
        roots.append(site.getusersitepackages())
    except Exception:
        pass
    for root in roots:
        for egg in Path(root).glob("MultiScaleDeformableAttention*.egg"):
            if str(egg) not in sys.path:
                sys.path.insert(0, str(egg))
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        return None


if find_extension() is not None:
    print("deformable attention: compiled CUDA extension")
    USING_FALLBACK = False
else:
    src = FUNC.read_text()
    if "_PurePythonMSDA" not in src:
        if UPSTREAM_IMPORT not in src:
            raise SystemExit(
                "the upstream import line moved -- open "
                "models/ops/functions/ms_deform_attn_func.py and patch it by hand")
        FUNC.write_text(src.replace(UPSTREAM_IMPORT, SHIM.strip(), 1))
    for name in [m for m in sys.modules if "ms_deform_attn" in m]:
        del sys.modules[name]
    print("deformable attention: pure-PyTorch fallback")
    print("  Same operator and the same numbers -- checked against an independent")
    print("  implementation of its definition to 3e-16 -- just slower. Fine to proceed.")
    USING_FALLBACK = True

# Load the module by path rather than by package: it has no relative imports, and
# this avoids dragging in the whole model stack just to check one operator.
spec = importlib.util.spec_from_file_location("_msda_check", FUNC)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# Prove the operator before trusting it. Shapes chosen so a wrong answer cannot
# accidentally have the right shape.
N, M, D, Lq, P = 2, 8, 32, 7, 4
shapes = torch.as_tensor([[8, 6], [4, 3]], dtype=torch.long, device="cuda")
sizes = shapes[:, 0] * shapes[:, 1]
starts = torch.cat([sizes.new_zeros(1), sizes.cumsum(0)[:-1]])
with torch.no_grad():
    out = mod.MSDeformAttnFunction.apply(
        torch.randn(N, int(sizes.sum()), M, D, device="cuda"), shapes, starts,
        torch.rand(N, Lq, M, len(shapes), P, 2, device="cuda"),
        torch.rand(N, Lq, M, len(shapes), P, device="cuda"), 64)
assert tuple(out.shape) == (N, Lq, M * D), f"got {tuple(out.shape)}, expected {(N, Lq, M * D)}"
assert torch.isfinite(out).all(), "the operator returned non-finite values"
print(f"   smoke test passed: {tuple(out.shape)}")

## 5. Our plans

Pulled straight from the golden set's own recorded URLs, so nothing needs uploading and
nothing needs to exist on your machine. 25 small PNGs.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/romainbigare/visit-it.git 2>/dev/null || echo "already cloned"

import json, urllib.request, ssl
from pathlib import Path

PLANS = Path("/content/plans/raw"); PLANS.mkdir(parents=True, exist_ok=True)
golden = json.loads(Path("/content/visit-it/data/golden/golden_set.json").read_text())

ctx = ssl.create_default_context()
got, missing = [], []
for listing in golden["listings"]:
    plans = listing.get("floorplans") or []
    if not plans:
        missing.append(listing["listing_id"]); continue
    dest = PLANS / f"{listing['listing_id']}.png"
    if not dest.exists():
        try:
            req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})
            dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())
        except Exception as exc:
            print("  failed", listing["listing_id"], exc); missing.append(listing["listing_id"]); continue
    got.append(listing["listing_id"])

print(f"{len(got)} plans downloaded, {len(missing)} listings have no plan: {missing}")

## 6. Three versions of each plan

Raster2Seq was trained on CubiCasa5K, which is dark ink on white paper with no estate-agent
captions. Our plans are neither, and we already know from the wall model that the mismatch
matters: on one tinted plan it labelled 62% of the flat "window" until the page was
levelled to white.

So we prepare three variants and run all three. If **raw** wins, the model is robust and we
can drop the preprocessing; if **clean** wins, the preprocessing is load-bearing and belongs
in the pipeline. Either answer is worth having.

| variant | what it is |
|---|---|
| `raw` | the plan exactly as the agent published it |
| `white` | page levelled to white, darkest ink to black |
| `clean` | levelled, and every OCR word box painted out |

In [ ]:
pip_install("pytesseract")
!apt-get -qq install -y tesseract-ocr > /dev/null

import cv2, numpy as np, pytesseract
from pathlib import Path
from PIL import Image

RAW = Path("/content/plans/raw")
VARIANTS = {name: Path(f"/content/plans/{name}") for name in ("white", "clean")}
for d in VARIANTS.values():
    d.mkdir(parents=True, exist_ok=True)


def load_rgb(path):
    '''Composite transparency onto white -- several agent plans are LA-mode PNGs
    whose ink lives entirely in the alpha channel, and a naive convert to RGB
    turns them solid black.'''
    im = Image.open(path)
    if im.mode in ("LA", "RGBA", "P"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return np.array(im.convert("RGB"))


def whiten(rgb):
    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))
    if page < 110:                      # light ink on a dark page
        lum, page = 255 - lum, 255 - page
    floor = float(np.percentile(lum, 1))
    if page - floor < 20:
        return cv2.cvtColor(lum, cv2.COLOR_GRAY2RGB)
    out = np.clip((lum.astype(np.float32) - floor) * (255.0 / (page - floor)), 0, 255)
    return cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_GRAY2RGB)


def blank_text(rgb):
    data = pytesseract.image_to_data(rgb, output_type=pytesseract.Output.DICT)
    out = rgb.copy()
    h, w = rgb.shape[:2]
    for i, conf in enumerate(data["conf"]):
        if float(conf) < 40 or not data["text"][i].strip():
            continue
        x, y = data["left"][i] - 2, data["top"][i] - 2
        out[max(0, y):min(h, y + data["height"][i] + 4),
            max(0, x):min(w, x + data["width"][i] + 4)] = 255
    return out


for src in sorted(RAW.glob("*.png")):
    rgb = load_rgb(src)
    Image.fromarray(rgb).save(src)                     # normalise the raw copy too
    w = whiten(rgb)
    Image.fromarray(w).save(VARIANTS["white"] / src.name)
    Image.fromarray(blank_text(w)).save(VARIANTS["clean"] / src.name)
print("prepared", len(list(RAW.glob('*.png'))), "plans in 3 variants")

## 7. Predict

`predict.py` walks a directory for images, so each variant is just a directory. The flags
are the authors' own `tools/predict_cc5k.sh` verbatim except for the paths — deliberately,
so a bad result here is the model's and not our misconfiguration.

`--disable_poly_refine` matches their published CubiCasa5K numbers, and also means the
differentiable rasteriser is not needed even if it failed to build in §4.

In [ ]:
import subprocess, time

FLAGS = [
    "--dataset_name=cubicasa", "--checkpoint=hf:cubicasa5k",
    "--semantic_classes=12", "--input_channels", "3",
    "--poly2seq", "--seq_len", "512", "--num_bins", "32",
    "--disable_poly_refine", "--dec_attn_concat_src",
    "--per_token_sem_loss", "--use_anchor", "--ema4eval", "--save_pred",
]

%cd /content/Raster2Seq
if USING_FALLBACK:
    print("running on the pure-PyTorch operator — about a minute per variant\n")
for variant in ("raw", "white", "clean"):
    t0 = time.time()
    cmd = ["python", "predict.py", f"--dataset_root=/content/plans/{variant}",
           f"--output_dir=/content/preds/{variant}"] + FLAGS
    r = subprocess.run(cmd, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-6:]
    print(f"--- {variant}: exit {r.returncode} in {time.time()-t0:.0f}s")
    for line in tail:
        print("   ", line)

## 8. Put the polygons back in our coordinates

The model works on a 256×256 letterboxed copy, so the polygons come out in *that* space.
Undoing it is exact — resize by `min(256/h, 256/w)`, centre-pad, so invert in that order.

This matters more than it sounds: the last time a coordinate space was assumed rather than
inverted, every outline in the review images was off by a couple of percent and correct
rooms looked like they floated clear of their walls.

In [ ]:
import json
from pathlib import Path
import numpy as np
from PIL import Image

CC5K_LABEL = {0: "Outdoor", 1: "Kitchen", 2: "Living Room", 3: "Bed Room", 4: "Bath",
              5: "Entry", 6: "Storage", 7: "Garage", 8: "Undefined", 9: "Window", 10: "Door"}
ROOM_CLASSES = set(range(0, 9))     # 9 and 10 are window and door instances
SIZE = 256


def to_source_pixels(poly, src_w, src_h, size=SIZE):
    '''Invert ResizeAndPad: undo the centre pad, then the aspect-preserving resize.'''
    scale = min(size / src_h, size / src_w)
    new_h, new_w = int(src_h * scale), int(src_w * scale)
    left, top = (size - new_w) // 2, (size - new_h) // 2
    p = np.asarray(poly, dtype=float).reshape(-1, 2)
    return np.stack([(p[:, 0] - left) / scale, (p[:, 1] - top) / scale], axis=1)


def collect(variant):
    root = Path(f"/content/preds/{variant}")
    jsons = sorted(root.rglob("jsons/*.json"))
    out = {}
    for jf in jsons:
        lid = jf.stem
        src = Image.open(f"/content/plans/{variant}/{lid}.png")
        rooms, apertures = [], []
        for inst in json.loads(jf.read_text()):
            poly = to_source_pixels(inst["segmentation"], src.width, src.height)
            rec = {"category_id": inst["category_id"],
                   "label": CC5K_LABEL.get(inst["category_id"], "?"),
                   "polygon_px": poly.tolist()}
            (rooms if inst["category_id"] in ROOM_CLASSES else apertures).append(rec)
        out[lid] = {"listing_id": lid, "image_size_px": [src.width, src.height],
                    "rooms": rooms, "apertures": apertures}
    return out


results = {v: collect(v) for v in ("raw", "white", "clean")}
for v, r in results.items():
    n_plans = len(r)
    n_rooms = sum(len(x["rooms"]) for x in r.values())
    n_ap = sum(len(x["apertures"]) for x in r.values())
    per = f"{n_rooms / n_plans:.1f}" if n_plans else "-"
    print(f"{v:6s}  {n_plans:2d} plans  {n_rooms:3d} rooms ({per} each)  {n_ap:3d} doors+windows")

## 9. Score it the same way we score ourselves

The metric from `docs/PLAN-READING-REPORT.md`: walk each room outline and ask how much of it
lies on a **wall** — using the CubiCasa wall segmenter as the wall reference, because
scoring against "any drawn line" cannot tell a room that traces its walls from one that
traces the kitchen cabinets.

Two caveats worth keeping in view while you read the table. Raster2Seq predicts polygons at
256×256, so its outlines are coarser than ours by construction and it is mildly penalised
on a fit metric measured at full resolution. And the wall reference is itself a prediction,
so this favours nobody in particular but is not ground truth. **The pictures in §10 are the
real verdict.**

In [ ]:
pip_install("segmentation-models-pytorch", "safetensors")
import sys
sys.path.insert(0, "/content/visit-it")

from pathlib import Path
import numpy as np
from pipeline.floorplan import wallnet

Path("/content/visit-it/models").mkdir(exist_ok=True)
wallnet.MODEL_PATH = Path("/content/visit-it/models/plan_walls.safetensors")
if not wallnet.MODEL_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)
print("wall reference available:", wallnet.available())

In [ ]:
import cv2

def wall_reference(rgb):
    '''The same barrier stage 5 uses, so the score means the same thing.'''
    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))
    ink = (lum < max(30, page - 40)).astype(np.uint8)
    return wallnet.barrier(rgb, ink, [])


refs, scores = {}, {}
for lid in sorted(results["raw"]):
    rgb = np.array(Image.open(f"/content/plans/raw/{lid}.png").convert("RGB"))
    refs[lid] = wall_reference(rgb)

for variant, byid in results.items():
    per_plan = {}
    for lid, rec in byid.items():
        ref = refs.get(lid)
        if ref is None or not ref.any():
            continue
        polys = [r["polygon_px"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        s = wallnet.outline_on_wall(polys, ref)
        if s:
            per_plan[lid] = s
    scores[variant] = per_plan
    flat = [v for s in per_plan.values() for v in s]
    if flat:
        print(f"{variant:6s}  {len(flat):3d} rooms  median outline-on-wall {np.median(flat):.3f}"
              f"   >=0.8 {sum(1 for v in flat if v >= .8) / len(flat):5.1%}")

print()
print("current pipeline, same metric, same plans:  median 0.827   >=0.8 55.9%")

## 10. Look at them

Score tables have already misled us once on this exact question. Read the pictures.

For each plan: the plan on the left, Raster2Seq's rooms shaded and named on the right. What
to look for specifically —

- does an open-plan **"RECEPTION / DINING ROOM" come out as one room** (ours splits it)
- do outlines **stop at kitchen cabinets** or run to the wall behind them
- are **door swing arcs** treated as boundaries
- are **bathrooms, WCs and hallways** found at all — ours often misses them
- are the **room types** right, without needing the caption text

In [ ]:
import colorsys
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly

BEST = "clean"   # change to "raw" or "white" once §9 says which one won


def hue(i):
    r, g, b = colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)
    return (r, g, b)


byid = results[BEST]
ids = sorted(byid)
for lid in ids:
    rec = byid[lid]
    rgb = np.array(Image.open(f"/content/plans/raw/{lid}.png").convert("RGB"))
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    for ax in axes:
        ax.imshow(rgb); ax.axis("off")
    for i, room in enumerate(rec["rooms"]):
        p = np.asarray(room["polygon_px"])
        axes[1].add_patch(MplPoly(p, closed=True, facecolor=hue(i) + (0.35,),
                                  edgecolor=hue(i), linewidth=2))
        axes[1].text(*p.mean(axis=0), room["label"], ha="center", va="center",
                     fontsize=9, weight="bold",
                     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))
    for ap in rec["apertures"]:
        p = np.asarray(ap["polygon_px"])
        axes[1].add_patch(MplPoly(p, closed=True, facecolor="none",
                                  edgecolor="crimson" if ap["label"] == "Door" else "royalblue",
                                  linewidth=2))
    med = np.median(scores[BEST].get(lid, [0]))
    axes[0].set_title(f"{lid} — the plan", fontsize=11)
    axes[1].set_title(f"Raster2Seq · {len(rec['rooms'])} rooms · "
                      f"{len(rec['apertures'])} doors/windows · outline-on-wall {med:.2f}",
                      fontsize=11)
    plt.tight_layout(); plt.show()

## 11. Take the results home

A zip with the polygons in our own pixel coordinates and the scores, so the comparison can
be rerun and, if the answer is yes, wired into stage 5 as another engine behind the same
`AD-4` interface — no artifact contract has to move.

In [ ]:
import json, shutil
from pathlib import Path

OUT = Path("/content/raster2seq_results"); OUT.mkdir(exist_ok=True)
for variant, byid in results.items():
    (OUT / f"{variant}.json").write_text(json.dumps(byid, indent=1))

summary = {}
for variant, per_plan in scores.items():
    flat = [v for s in per_plan.values() for v in s]
    summary[variant] = {
        "rooms": len(flat),
        "median_outline_on_wall": float(np.median(flat)) if flat else None,
        "fraction_at_or_above_0.8": (sum(1 for v in flat if v >= .8) / len(flat)) if flat else None,
        "per_listing_median": {k: float(np.median(v)) for k, v in per_plan.items()},
    }
summary["baseline_current_pipeline"] = {"median_outline_on_wall": 0.827,
                                        "fraction_at_or_above_0.8": 0.559}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1))

shutil.make_archive("/content/raster2seq_results", "zip", OUT)
from google.colab import files
files.download("/content/raster2seq_results.zip")

## What the answer means

**If the pictures look right** — open-plan spaces intact, outlines on walls not cabinets,
bathrooms and hallways found, room types correct — then Raster2Seq becomes stage 5's
primary engine and the watershed drops to a fallback. That also removes the need to
fine-tune the wall model at all, because there would no longer be a wall model in the
critical path. Drop the zip in the repo and say so.

**If they look wrong in a consistent way** — say every room is there but the polygons are
too coarse at 256×256 — that is a resolution problem, not a capability problem, and the
authors publish a 512-resolution `Raster2Graph-512` checkpoint worth trying next.

**If they look wrong in an inconsistent way**, the domain gap is real and the answer is the
same one as for the wall model: fine-tune on our own plans. That is
`notebooks/finetune_wallnet_colab.ipynb`.

---

*If §4c reported the pure-PyTorch fallback, the predictions are identical — it is the same
operator, checked against an independent implementation of its definition to 3e-16 — but
it runs several times slower. That affects how long you waited, not what you are looking
at.*